# Retry bad sector

This notebook loads `BAD` entries from `diskimage.iso.copy.txt`, retries reading one bad region from the Blu-ray drive, and lets you patch `diskimage.iso` only after a successful manual read.

In [8]:
from pathlib import Path
import hashlib
import time

from bd_utils import open_bd_drive
from compare_bd_iso import format_bytes
from create_iso_from_bd import get_copy_progress_path, parse_copy_progress

DEVICE = "/dev/sr1"
ISO_PATH = Path("diskimage.iso")
PROGRESS_PATH = Path(get_copy_progress_path(str(ISO_PATH)))
OUTPUT_DIR = Path("manual_retry_chunks")
OUTPUT_DIR.mkdir(exist_ok=True)

RETRY_COUNT = 30
RETRY_DELAY_SECONDS = 0.5
REOPEN_EACH_ATTEMPT = True

print(f"Device: {DEVICE}")
print(f"ISO: {ISO_PATH.resolve()}")
print(f"Progress log: {PROGRESS_PATH.resolve()}")

Device: /dev/sr1
ISO: /media/qidai/MP600/bd_iso/diskimage.iso
Progress log: /media/qidai/MP600/bd_iso/diskimage.iso.copy.txt


In [2]:
progress = parse_copy_progress(PROGRESS_PATH)

bad_regions = []
seen = set()
for offset, size in progress.bad_regions:
    if (offset, size) not in seen:
        seen.add((offset, size))
        bad_regions.append((offset, size))

if not bad_regions:
    raise RuntimeError("No BAD entries found in the copy progress log.")

for index, (offset, size) in enumerate(bad_regions):
    sector = offset // 2048
    aligned = "yes" if offset % 2048 == 0 and size % 2048 == 0 else "no"
    print(
        f"[{index}] offset={offset} (0x{offset:X}), "
        f"size={size} ({format_bytes(size)}), sector={sector}, 2048-aligned={aligned}"
    )

SELECTED_BAD_INDEX = -1
OFFSET, SIZE = bad_regions[SELECTED_BAD_INDEX]
print(f"\nSelected BAD region: offset={OFFSET} (0x{OFFSET:X}), size={SIZE}")

[0] offset=3506438144 (0xD1000000), size=2048 (2.000 KB), sector=1712128, 2048-aligned=yes

Selected BAD region: offset=3506438144 (0xD1000000), size=2048


In [9]:
def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def hexdump(data, base_offset=0, limit=256):
    shown = data[:limit]
    for row_start in range(0, len(shown), 16):
        row = shown[row_start : row_start + 16]
        hex_text = " ".join(f"{byte:02X}" for byte in row)
        ascii_text = "".join(chr(byte) if 32 <= byte <= 126 else "." for byte in row)
        print(f"{base_offset + row_start:08X}  {hex_text:<48}  {ascii_text}")
    if len(data) > limit:
        print(f"... {len(data) - limit} more bytes not shown")


def read_exact(handle, offset, size):
    handle.seek(offset)
    data = handle.read(size)
    if len(data) != size:
        raise IOError(f"short read: expected {size} bytes, got {len(data)}")
    return data


def read_at_most(handle, offset, size):
    handle.seek(offset)
    return handle.read(size)


def read_iso_region(offset=OFFSET, size=SIZE):
    iso_size = ISO_PATH.stat().st_size
    with ISO_PATH.open("rb") as handle:
        data = read_at_most(handle, offset, size)
    if len(data) != size:
        missing = size - len(data)
        if offset >= iso_size:
            print(
                f"ISO has no bytes at this offset yet: offset={offset}, "
                f"iso_size={iso_size}, missing={missing}"
            )
        else:
            print(
                f"ISO region is partial: expected {size} bytes, "
                f"got {len(data)}, missing={missing}"
            )
    return data


def read_bd_region(offset=OFFSET, size=SIZE):
    with open_bd_drive(DEVICE) as handle:
        return read_exact(handle, offset, size)


def retry_bad_region(offset=OFFSET, size=SIZE, attempts=RETRY_COUNT, delay=RETRY_DELAY_SECONDS):
    successes = []
    last_error = None

    if REOPEN_EACH_ATTEMPT:
        for attempt in range(1, attempts + 1):
            try:
                data = read_bd_region(offset, size)
                digest = sha256_bytes(data)
                successes.append(data)
                print(f"[ok] attempt {attempt}: read {len(data)} bytes, sha256={digest}")
                time.sleep(delay)
            except Exception as exc:
                last_error = exc
                print(f"[fail] attempt {attempt}: {exc}")
                time.sleep(delay)
            if attempt < attempts:
                time.sleep(delay)
    else:
        with open_bd_drive(DEVICE) as handle:
            for attempt in range(1, attempts + 1):
                try:
                    data = read_exact(handle, offset, size)
                    digest = sha256_bytes(data)
                    successes.append(data)
                    print(f"[ok] attempt {attempt}: read {len(data)} bytes, sha256={digest}")
                except Exception as exc:
                    last_error = exc
                    print(f"[fail] attempt {attempt}: {exc}")
                if attempt < attempts:
                    time.sleep(delay)

    if not successes:
        raise RuntimeError(f"No successful reads after {attempts} attempts. Last error: {last_error}")

    unique = {sha256_bytes(data): data for data in successes}
    print(f"\nSuccessful reads: {len(successes)}; unique payloads: {len(unique)}")
    for digest, data in unique.items():
        path = OUTPUT_DIR / f"bad_{offset}_{size}_{digest[:12]}.bin"
        path.write_bytes(data)
        print(f"Saved {digest[:12]} to {path}")

    if len(unique) > 1:
        print("Multiple different reads succeeded. Inspect them before patching the ISO.")

    return successes[-1]


print("Helpers loaded.")

Helpers loaded.


In [4]:
iso_before = read_iso_region()
print(f"Current ISO bytes read: {len(iso_before)} / {SIZE}")
if iso_before:
    print(f"Current ISO bytes sha256: {sha256_bytes(iso_before)}")
    hexdump(iso_before, base_offset=OFFSET)
else:
    print("Nothing to hexdump yet. This bad region starts at the current ISO EOF.")

ISO has no bytes at this offset yet: offset=3506438144, iso_size=3506438144, missing=2048
Current ISO bytes read: 0 / 2048
Nothing to hexdump yet. This bad region starts at the current ISO EOF.


In [5]:
CHUNK_SIZE = 4194304

In [11]:
next_chunk_offset

3510632448

In [10]:
next_chunk_offset = offset + CHUNK_SIZE
next_chunk = retry_bad_region(next_chunk_offset, CHUNK_SIZE)
print(f"\nNext chunk read: {len(next_chunk)} bytes, sha256={sha256_bytes(next_chunk)}")
hexdump(next_chunk, base_offset=next_chunk_offset)

[fail] attempt 1: [Errno 5] Input/output error
[fail] attempt 2: [Errno 5] Input/output error
[ok] attempt 3: read 4194304 bytes, sha256=4ca8c5cc168d5f8956285077e93c846c3dd9f67bdc94efcd13cb616be26dd903
[ok] attempt 4: read 4194304 bytes, sha256=4ca8c5cc168d5f8956285077e93c846c3dd9f67bdc94efcd13cb616be26dd903
[ok] attempt 5: read 4194304 bytes, sha256=4ca8c5cc168d5f8956285077e93c846c3dd9f67bdc94efcd13cb616be26dd903
[ok] attempt 6: read 4194304 bytes, sha256=4ca8c5cc168d5f8956285077e93c846c3dd9f67bdc94efcd13cb616be26dd903
[ok] attempt 7: read 4194304 bytes, sha256=4ca8c5cc168d5f8956285077e93c846c3dd9f67bdc94efcd13cb616be26dd903
[ok] attempt 8: read 4194304 bytes, sha256=4ca8c5cc168d5f8956285077e93c846c3dd9f67bdc94efcd13cb616be26dd903
[ok] attempt 9: read 4194304 bytes, sha256=4ca8c5cc168d5f8956285077e93c846c3dd9f67bdc94efcd13cb616be26dd903
[ok] attempt 10: read 4194304 bytes, sha256=4ca8c5cc168d5f8956285077e93c846c3dd9f67bdc94efcd13cb616be26dd903
[ok] attempt 11: read 4194304 bytes, sha2

KeyboardInterrupt: 

In [9]:
bd_retry = retry_bad_region()
print(f"\nLast successful BD read sha256: {sha256_bytes(bd_retry)}")
hexdump(bd_retry, base_offset=OFFSET)

[ok] attempt 1: read 2048 bytes, sha256=fab33b5a085b9c93941d0eb73de5a291b2d5ec4f53456d6671ee69ab5cb7a468
[ok] attempt 2: read 2048 bytes, sha256=fab33b5a085b9c93941d0eb73de5a291b2d5ec4f53456d6671ee69ab5cb7a468
[ok] attempt 3: read 2048 bytes, sha256=fab33b5a085b9c93941d0eb73de5a291b2d5ec4f53456d6671ee69ab5cb7a468
[ok] attempt 4: read 2048 bytes, sha256=fab33b5a085b9c93941d0eb73de5a291b2d5ec4f53456d6671ee69ab5cb7a468
[ok] attempt 5: read 2048 bytes, sha256=fab33b5a085b9c93941d0eb73de5a291b2d5ec4f53456d6671ee69ab5cb7a468
[ok] attempt 6: read 2048 bytes, sha256=fab33b5a085b9c93941d0eb73de5a291b2d5ec4f53456d6671ee69ab5cb7a468
[ok] attempt 7: read 2048 bytes, sha256=fab33b5a085b9c93941d0eb73de5a291b2d5ec4f53456d6671ee69ab5cb7a468
[ok] attempt 8: read 2048 bytes, sha256=fab33b5a085b9c93941d0eb73de5a291b2d5ec4f53456d6671ee69ab5cb7a468
[ok] attempt 9: read 2048 bytes, sha256=fab33b5a085b9c93941d0eb73de5a291b2d5ec4f53456d6671ee69ab5cb7a468
[ok] attempt 10: read 2048 bytes, sha256=fab33b5a085b9c

In [10]:
if len(iso_before) != SIZE:
    print(
        f"Current ISO does not contain a full region here yet: "
        f"{len(iso_before)} / {SIZE} bytes present."
    )
else:
    print(f"BD retry matches current ISO bytes: {bd_retry == iso_before}")

if len(iso_before) == SIZE and bd_retry != iso_before:
    diffs = [(i, a, b) for i, (a, b) in enumerate(zip(bd_retry, iso_before)) if a != b]
    print(f"Different bytes: {len(diffs)}")
    for relative_offset, bd_byte, iso_byte in diffs[:20]:
        absolute_offset = OFFSET + relative_offset
        print(
            f"0x{absolute_offset:X}: BD=0x{bd_byte:02X}, ISO=0x{iso_byte:02X}"
        )
    if len(diffs) > 20:
        print(f"... {len(diffs) - 20} more differences")

Current ISO does not contain a full region here yet: 0 / 2048 bytes present.


## Patch the ISO

Run the next cell only after `bd_retry` is a full-size read you trust. It writes exactly `SIZE` bytes at `OFFSET` and creates a backup of the old ISO bytes first.

In [11]:
CONFIRM_PATCH = True

if not CONFIRM_PATCH:
    raise RuntimeError("Set CONFIRM_PATCH = True in this cell when you are ready to patch the ISO.")
if len(bd_retry) != SIZE:
    raise RuntimeError(f"Refusing to patch: expected {SIZE} bytes, got {len(bd_retry)}")

backup_path = OUTPUT_DIR / f"iso_before_{OFFSET}_{len(iso_before)}_{sha256_bytes(iso_before)[:12]}.bin"
backup_path.write_bytes(iso_before)

with ISO_PATH.open("r+b") as handle:
    handle.seek(OFFSET)
    handle.write(bd_retry)
    handle.flush()

iso_after = read_iso_region()
print(f"Patched {SIZE} bytes at offset {OFFSET} (0x{OFFSET:X}).")
print(f"Backup of previous ISO bytes saved to {backup_path} ({len(iso_before)} bytes)")
print(f"ISO after sha256: {sha256_bytes(iso_after)}")
print(f"Patch verified in ISO: {iso_after == bd_retry}")

Patched 2048 bytes at offset 3506440192 (0xD1000800).
Backup of previous ISO bytes saved to manual_retry_chunks/iso_before_3506440192_0_e3b0c44298fc.bin (0 bytes)
ISO after sha256: fab33b5a085b9c93941d0eb73de5a291b2d5ec4f53456d6671ee69ab5cb7a468
Patch verified in ISO: True


In [12]:
from create_iso_from_bd import write_copy_progress

new_checkpoint = OFFSET + SIZE
iso_size = ISO_PATH.stat().st_size

if iso_size < new_checkpoint:
    raise RuntimeError(
        f"Refusing to update log: ISO size is {iso_size}, "
        f"but checkpoint would be {new_checkpoint}."
    )
if read_iso_region() != bd_retry:
    raise RuntimeError("Refusing to update log: patched ISO bytes do not match bd_retry.")

progress = parse_copy_progress(PROGRESS_PATH)
old_checkpoint = progress.checkpoint
progress.checkpoint = max(progress.checkpoint, new_checkpoint)
write_copy_progress(PROGRESS_PATH, progress)

print(f"Updated {PROGRESS_PATH}")
print(f"Checkpoint: {old_checkpoint} -> {progress.checkpoint}")
print("BAD entries were preserved for the repair record.")

Updated diskimage.iso.copy.txt
Checkpoint: 3506440192 -> 3506442240
BAD entries were preserved for the repair record.
